# SpaceX Launch Sites Location Analysis: Mapping the Journey

This notebook is a visual chapter in my end-to-end SpaceX Falcon 9 data science project. After collecting, cleaning, and analyzing launch data, I now explore the geography of SpaceX launch sites. Mapping and spatial analysis help reveal how location influences mission outcomes, adding a new dimension to the story.

## Project Narrative

My project follows a real-world data science pipeline:

1. **Web Scraping & Data Collection**: Gather launch records from Wikipedia and SpaceX APIs.
2. **Data Wrangling & Cleaning**: Prepare and clean the raw data for analysis.
3. **SQL Analysis & Feature Engineering**: Use SQL and Python to explore and transform the data.
4. **Exploratory Data Analysis (EDA)**: Visualize trends and patterns.
5. **Geospatial Analysis & Mapping** (this notebook): Analyze and visualize launch site locations and their impact.
6. **Machine Learning & Prediction**: Build and compare models to predict Falcon 9 first stage landing outcomes.
7. **Dashboarding & Communication**: Present findings with interactive dashboards and clear visualizations.

This notebook brings the data to life on the map, connecting numbers to real-world places and decisions.

## Objectives
- Mark all launch sites on an interactive map
- Visualize successful and failed launches for each site
- Calculate distances from launch sites to nearby features

Let's start by importing the required Python packages:

In [ ]:
!pip3 install wget
import folium
import wget
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

## Task 1: Mark all launch sites on a map

In [ ]:
# Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = pd.read_csv(spacex_csv_file)

In [ ]:
# Select relevant sub-columns: `Launch Site`, `Lat`, `Long`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

In [ ]:
# Create a folium map centered at NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add Circle and Marker for each launch site
for _, row in launch_sites_df.iterrows():
    coordinates = [row['Lat'], row['Long']]
    folium.Circle(
        location=coordinates,
        radius=50,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.Marker(
        location=coordinates,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site']
        )
    ).add_to(site_map)
site_map

## Task 2: Mark the success/failed launches for each site on the map

In [ ]:
# Assign marker color based on class value
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Initialize marker cluster
marker_cluster = MarkerCluster().add_to(site_map)

# Add markers for each launch
for _, record in spacex_df.iterrows():
    folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=record['Launch Site']
    ).add_to(marker_cluster)
site_map

## Task 3: Calculate the distances between a launch site and its proximities

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# Example: Calculate distance to closest coastline
launch_site_lat = 28.563197
launch_site_lon = -80.576820
coastline_lat = 28.562997
coastline_lon = -80.56760
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(distance_coastline)

# Add marker and polyline for visualization
coordinate = (coastline_lat, coastline_lon)
distance_marker = folium.Marker(
  coordinate,
  icon=DivIcon(
      icon_size=(20,20),
      icon_anchor=(0,0),
      html='<div style="font-size: 12; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_coastline),
      )
  )
distance_marker.add_to(site_map)
folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], color='skyblue').add_to(site_map)
site_map